In [1]:
pwd

'/Users/prasannanekarkaleya/Work/Trades/files'

In [2]:
"""
NSE F&O Market Scanner — v4
─────────────────────────────
Changes in v4:
  • yfinance-only data source (Bhavcopy removed for simplicity)
  • Corporate Actions module integrated (auto-discovers via NSE API)
  • Risky events FLAGGED in output (per user preference: no auto-quarantine)
  • Scan runs as before; output prepended with corp action summary
"""

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Local import — must be in same folder
from corporate_actions import (
    update_corporate_actions,
    get_quarantined_symbols,
    get_flagged_symbols,
    get_upcoming_actions,
    format_flag_report,
)


# ─────────────────────────────────────────────
# NSE SECTOR MAPPING
# ─────────────────────────────────────────────
NSE_SECTOR_INDICES = {
    "Nifty Bank":               "^NSEBANK",
    "Nifty Private Bank":       "NIFTY_PVT_BANK.NS",
    "Nifty PSU Bank":           "^CNXPSUBANK",
    "Nifty Financial Services": "NIFTY_FIN_SERVICE.NS",
    "Nifty IT":                 "^CNXIT",
    "Nifty Pharma":             "^CNXPHARMA",
    "Nifty Healthcare":         "NIFTY_HEALTHCARE.NS",
    "Nifty Auto":               "^CNXAUTO",
    "Nifty FMCG":               "^CNXFMCG",
    "Nifty Metal":              "^CNXMETAL",
    "Nifty Realty":             "^CNXREALTY",
    "Nifty Media":              "^CNXMEDIA",
    "Nifty Chemicals":          "NIFTY_CHEM.NS",
    "Nifty Consumer Durables":  "NIFTY_CONSR_DURBL.NS",
    "Nifty Energy":             "^CNXENERGY",
    "Nifty Infra":              "^CNXINFRA",
    "Nifty Oil & Gas":          "NIFTY_OIL_AND_GAS.NS",
}

SECTOR_STOCKS = {
    "Nifty Bank": [
        "HDFCBANK", "ICICIBANK", "KOTAKBANK", "AXISBANK", "SBIN",
        "INDUSINDBK", "BANDHANBNK", "FEDERALBNK", "IDFCFIRSTB",
        "PNB", "BANKBARODA", "AUBANK"
    ],
    "Nifty Private Bank": [
        "HDFCBANK", "ICICIBANK", "KOTAKBANK", "AXISBANK", "INDUSINDBK",
        "IDFCFIRSTB", "FEDERALBNK", "BANDHANBNK", "AUBANK", "RBLBANK"
    ],
    "Nifty PSU Bank": [
        "SBIN", "BANKBARODA", "PNB", "CANBK", "UNIONBANK",
        "INDIANB", "BANKINDIA", "IOB", "MAHABANK", "CENTRALBK"
    ],
    "Nifty Financial Services": [
        "HDFCBANK", "ICICIBANK", "SBIN", "BAJFINANCE", "AXISBANK",
        "KOTAKBANK", "BAJAJFINSV", "SBILIFE", "HDFCLIFE", "SHRIRAMFIN",
        "JIOFIN", "PFC", "MUTHOOTFIN", "ICICIGI", "ICICIPRULI",
        "CHOLAFIN", "RECLTD", "LICI", "M&MFIN", "SBICARD"
    ],
    "Nifty IT": [
        "TCS", "INFY", "WIPRO", "HCLTECH", "TECHM",
        "LTIM", "MPHASIS", "PERSISTENT", "COFORGE", "OFSS"
    ],
    "Nifty Pharma": [
        "SUNPHARMA", "DRREDDY", "CIPLA", "DIVISLAB", "BIOCON",
        "AUROPHARMA", "ALKEM", "TORNTPHARM", "LUPIN", "IPCALAB",
        "ZYDUSLIFE", "GLENMARK", "LAURUSLABS", "GRANULES"
    ],
    "Nifty Healthcare": [
        "SUNPHARMA", "DRREDDY", "CIPLA", "DIVISLAB", "APOLLOHOSP",
        "MAXHEALTH", "FORTIS", "BIOCON", "ALKEM", "TORNTPHARM",
        "LUPIN", "LAURUSLABS", "ZYDUSLIFE", "SYNGENE"
    ],
    "Nifty Auto": [
        "MARUTI", "TMPV", "M&M", "BAJAJ-AUTO", "HEROMOTOCO",
        "EICHERMOT", "ASHOKLEY", "TVSMOTOR", "MOTHERSON", "BALKRISIND",
        "BHARATFORG", "BOSCHLTD", "EXIDEIND"
        # TATAMOTORS → TMPV after Oct 2025 demerger
    ],
    "Nifty FMCG": [
        "HINDUNILVR", "ITC", "NESTLEIND", "BRITANNIA", "DABUR",
        "MARICO", "GODREJCP", "COLPAL", "TATACONSUM", "UBL", "VBL"
    ],
    "Nifty Metal": [
        "TATASTEEL", "JSWSTEEL", "HINDALCO", "VEDL", "SAIL",
        "NMDC", "COALINDIA", "NATIONALUM", "APLAPOLLO", "JINDALSTEL",
        "HINDCOPPER", "RATNAMANI"
    ],
    "Nifty Realty": [
        "DLF", "GODREJPROP", "OBEROIRLTY", "PRESTIGE", "PHOENIXLTD",
        "BRIGADE", "SOBHA", "MAHLIFE", "LODHA"
    ],
    "Nifty Media": [
        "ZEEL", "SUNTV", "PVRINOX", "NAZARA", "SAREGAMA", "TIPS"
    ],
    "Nifty Chemicals": [
        "PIDILITIND", "SRF", "UPL", "TATACHEM", "PIIND",
        "DEEPAKNTR", "AARTIIND", "GNFC", "NAVINFLUOR", "ATUL",
        "CLEAN", "VINATIORGA"
    ],
    "Nifty Consumer Durables": [
        "TITAN", "HAVELLS", "VOLTAS", "DIXON", "CROMPTON",
        "WHIRLPOOL", "RAJESHEXPO", "BATAINDIA", "KAJARIACER", "BLUESTARCO"
    ],
    "Nifty Energy": [
        "RELIANCE", "ONGC", "BPCL", "IOC", "NTPC",
        "POWERGRID", "ADANIGREEN", "TATAPOWER", "GAIL", "ADANIPOWER"
    ],
    "Nifty Infra": [
        "LT", "ADANIPORTS", "ULTRACEMCO", "GRASIM", "SHREECEM",
        "AMBUJACEM", "ACC", "SIEMENS", "ABB", "CUMMINSIND",
        "BEL", "HAL", "BHEL"
    ],
    "Nifty Oil & Gas": [
        "RELIANCE", "ONGC", "BPCL", "IOC", "HINDPETRO",
        "GAIL", "OIL", "PETRONET", "IGL", "MGL", "GUJGASLTD"
    ],
}


# ─────────────────────────────────────────────
# DATA FETCH (yfinance)
# ─────────────────────────────────────────────
def fetch_data(ticker_ns, period="2y", interval="1d"):
    try:
        ticker = f"{ticker_ns}.NS"
        df = yf.download(ticker, period=period, interval=interval,
                         progress=False, auto_adjust=True)
        if df is None or len(df) < 50:
            return None
        df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
        return df
    except Exception:
        return None


def fetch_multi_timeframe(ticker_ns):
    daily   = fetch_data(ticker_ns, period="2y",  interval="1d")
    weekly  = fetch_data(ticker_ns, period="5y",  interval="1wk")
    monthly = fetch_data(ticker_ns, period="10y", interval="1mo")
    return daily, weekly, monthly


# ─────────────────────────────────────────────
# INDICATORS
# ─────────────────────────────────────────────
def compute_rsi(series, period=14):
    delta = series.diff()
    gain  = delta.clip(lower=0)
    loss  = -delta.clip(upper=0)
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()
    rs  = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi


def compute_macd(series, fast=12, slow=26, signal=9):
    ema_fast    = series.ewm(span=fast, adjust=False).mean()
    ema_slow    = series.ewm(span=slow, adjust=False).mean()
    macd_line   = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    histogram   = macd_line - signal_line
    return macd_line, signal_line, histogram


def compute_bollinger(series, period=20, std_dev=2):
    sma   = series.rolling(period).mean()
    std   = series.rolling(period).std()
    upper = sma + std_dev * std
    lower = sma - std_dev * std
    bw    = (upper - lower) / sma
    return upper, lower, bw


def compute_atr(df, period=14):
    high  = df['High']
    low   = df['Low']
    close = df['Close']
    tr = pd.concat([
        high - low,
        (high - close.shift()).abs(),
        (low  - close.shift()).abs()
    ], axis=1).max(axis=1)
    return tr.ewm(com=period - 1, min_periods=period).mean()


def compute_ema(series, period):
    return series.ewm(span=period, adjust=False).mean()


# ─────────────────────────────────────────────
# PATTERNS
# ─────────────────────────────────────────────
def detect_cup_and_handle(df, min_cup_bars=20, max_cup_bars=60):
    try:
        closes = df['Close'].values
        n = len(closes)
        if n < max_cup_bars + 10:
            return False
        window = closes[-(max_cup_bars + 15):]
        left_rim_idx = np.argmax(window[:max_cup_bars])
        left_rim = window[left_rim_idx]
        cup_bottom = np.min(window[left_rim_idx:left_rim_idx + max_cup_bars])
        depth_pct  = (left_rim - cup_bottom) / left_rim
        if not (0.15 <= depth_pct <= 0.50):
            return False
        right_rim_region = window[left_rim_idx + min_cup_bars:]
        if len(right_rim_region) < 5:
            return False
        right_rim = np.max(right_rim_region[:10])
        if abs(right_rim - left_rim) / left_rim > 0.05:
            return False
        handle_region = window[-10:]
        handle_low    = np.min(handle_region)
        handle_depth  = (right_rim - handle_low) / right_rim
        if handle_depth > 0.15:
            return False
        current = closes[-1]
        return bool(current >= left_rim * 0.98)
    except Exception:
        return False


def detect_elliott_wave3(df, lookback=60):
    try:
        closes  = df['Close'].values[-lookback:]
        volumes = df['Volume'].values[-lookback:]
        n = len(closes)
        if n < 30:
            return False
        w1_region = closes[:n//3]
        w1_start  = closes[0]
        w1_high   = np.max(w1_region)
        w1_high_idx = np.argmax(w1_region)
        if w1_high <= w1_start:
            return False
        w1_move = w1_high - w1_start
        w2_region = closes[w1_high_idx:w1_high_idx + n//3]
        w2_low    = np.min(w2_region)
        retracement = (w1_high - w2_low) / w1_move
        if not (0.382 <= retracement <= 0.618):
            return False
        current = closes[-1]
        if current <= w1_high:
            return False
        w1_vol_avg = np.mean(volumes[:w1_high_idx + 1])
        w3_vol_avg = np.mean(volumes[w1_high_idx + n//3:])
        return bool(w3_vol_avg >= w1_vol_avg * 1.2)
    except Exception:
        return False


def detect_bollinger_squeeze_breakout(df, period=20):
    try:
        _, _, bw = compute_bollinger(df['Close'], period)
        bw = bw.dropna()
        if len(bw) < 30:
            return False
        recent_bw  = bw.iloc[-5:]
        prior_bw   = bw.iloc[-20:-5]
        squeeze_bw = prior_bw.min()
        was_squeezed = squeeze_bw == bw.iloc[-25:-5].min()
        is_expanding = recent_bw.iloc[-1] > recent_bw.iloc[0]
        price_above_mid = df['Close'].iloc[-1] > df['Close'].rolling(period).mean().iloc[-1]
        return bool(was_squeezed and is_expanding and price_above_mid)
    except Exception:
        return False


# ─────────────────────────────────────────────
# FIBONACCI / ENTRY
# ─────────────────────────────────────────────
def compute_fibonacci_levels(swing_low, swing_high):
    diff = swing_high - swing_low
    return {
        "0.0":   swing_high,
        "0.236": swing_high - 0.236 * diff,
        "0.382": swing_high - 0.382 * diff,
        "0.5":   swing_high - 0.5   * diff,
        "0.618": swing_high - 0.618 * diff,
        "0.786": swing_high - 0.786 * diff,
        "1.0":   swing_low,
        "1.272": swing_high + 0.272 * diff,
        "1.618": swing_high + 0.618 * diff,
    }


def determine_entry(df):
    current = float(df['Close'].iloc[-1])
    atr     = float(compute_atr(df).iloc[-1])
    swing_high = float(df['High'].iloc[-50:].max())
    swing_low  = float(df['Low'].iloc[-50:].min())
    fibs = compute_fibonacci_levels(swing_low, swing_high)

    near_breakout = current >= swing_high * 0.99
    near_50  = abs(current - fibs["0.5"])   / fibs["0.5"]   < 0.02
    near_618 = abs(current - fibs["0.618"]) / fibs["0.618"] < 0.02

    if near_50 or near_618:
        entry_type  = "Fibonacci Retracement"
        entry_low   = round(fibs["0.618"], 2)
        entry_high  = round(fibs["0.5"],   2)
        stop_loss   = round(fibs["0.786"], 2)
        target1     = round(fibs["0.0"],   2)
        target2     = round(fibs["1.618"], 2)
    elif near_breakout:
        entry_type  = "Breakout"
        entry_low   = round(current, 2)
        entry_high  = round(current * 1.005, 2)
        stop_loss   = round(current - atr,   2)
        target1     = round(current + 2 * atr, 2)
        target2     = round(current + 3 * atr, 2)
    else:
        entry_type  = "Watch"
        entry_low   = round(swing_high * 0.99, 2)
        entry_high  = round(swing_high * 1.005, 2)
        stop_loss   = round(swing_high - atr, 2)
        target1     = round(swing_high + 2 * atr, 2)
        target2     = round(swing_high + 3 * atr, 2)

    risk   = entry_high - stop_loss
    reward = target1   - entry_high
    rr     = round(reward / risk, 2) if risk > 0 else 0

    return {
        "entry_type": entry_type,
        "entry_zone": f"₹{entry_low} – ₹{entry_high}",
        "stop_loss":  f"₹{stop_loss}",
        "target1":    f"₹{target1}",
        "target2":    f"₹{target2}",
        "rr_ratio":   rr,
    }


# ─────────────────────────────────────────────
# CORE SCAN
# ─────────────────────────────────────────────
def scan_stock(ticker_ns, flagged_symbols_set=None):
    """
    Scan a single stock. If it's in the flagged set, attach a warning flag.
    """
    daily, weekly, monthly = fetch_multi_timeframe(ticker_ns)
    if daily is None or weekly is None or monthly is None:
        return None
    if len(daily) < 60 or len(weekly) < 20 or len(monthly) < 14:
        return None

    close_d = daily['Close']
    close_w = weekly['Close']
    close_m = monthly['Close']

    rsi_daily   = float(compute_rsi(close_d).iloc[-1])
    rsi_weekly  = float(compute_rsi(close_w).iloc[-1])
    rsi_monthly = float(compute_rsi(close_m).iloc[-1])

    score = 0
    score_detail = {}

    swing_high = float(daily['High'].iloc[-50:].max())
    current    = float(close_d.iloc[-1])
    c1 = current >= swing_high * 0.99
    score += int(c1); score_detail["Price≥Resistance"] = c1

    vol_today = float(daily['Volume'].iloc[-1])
    vol_avg20 = float(daily['Volume'].iloc[-21:-1].mean())
    c2 = vol_today >= vol_avg20 * 1.5
    score += int(c2); score_detail["Volume≥1.5x"] = c2

    c3 = detect_bollinger_squeeze_breakout(daily)
    score += int(c3); score_detail["BB Squeeze→Expansion"] = c3

    macd, signal, hist = compute_macd(close_d)
    c4 = bool(hist.iloc[-1] > 0 and hist.iloc[-2] <= 0)
    score += int(c4); score_detail["MACD Crossover"] = c4

    c5 = detect_cup_and_handle(daily)
    score += int(c5); score_detail["Cup & Handle"] = c5

    c6 = detect_elliott_wave3(daily)
    score += int(c6); score_detail["Elliott Wave 3"] = c6

    c7 = rsi_daily > 60
    score += int(c7); score_detail["RSI>60 (Daily)"] = c7

    gfs  = (rsi_monthly > 60 and rsi_weekly > 60 and 40 <= rsi_daily <= 45)
    agfs = (rsi_monthly > 60 and rsi_weekly > 60 and rsi_daily > 60)

    entry = determine_entry(daily)
    qualifies = (score >= 4) or gfs or agfs

    if not qualifies:
        return None

    corp_action_flag = (flagged_symbols_set is not None and
                       ticker_ns in flagged_symbols_set)

    return {
        "ticker":         ticker_ns,
        "current_price":  f"₹{round(current, 2)}",
        "breakout_score": f"{score}/7",
        "score_detail":   score_detail,
        "rsi_daily":      round(rsi_daily,   2),
        "rsi_weekly":     round(rsi_weekly,  2),
        "rsi_monthly":    round(rsi_monthly, 2),
        "gfs":            gfs,
        "agfs":           agfs,
        "corp_action_flag": corp_action_flag,
        **entry,
    }


# ─────────────────────────────────────────────
# MARKET / SECTOR
# ─────────────────────────────────────────────
def get_market_direction():
    results = {}
    indices = {"NIFTY": "^NSEI", "SENSEX": "^BSESN"}
    for name, ticker in indices.items():
        try:
            df = yf.download(ticker, period="1y", interval="1d",
                             progress=False, auto_adjust=True)
            if df is None or len(df) < 50:
                results[name] = "Unknown"
                continue
            df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
            close = df['Close']
            ema50  = float(compute_ema(close, 50).iloc[-1])
            ema200 = float(compute_ema(close, 200).iloc[-1]) if len(close) >= 200 else None
            current = float(close.iloc[-1])
            rsi = float(compute_rsi(close).iloc[-1])
            macd_line, sig_line, _ = compute_macd(close)
            macd_bull = float(macd_line.iloc[-1]) > float(sig_line.iloc[-1])
            above_50  = current > ema50
            above_200 = (current > ema200) if ema200 else True
            bull_points = sum([above_50, above_200, macd_bull, rsi > 50])
            if bull_points >= 3:
                direction = "📈 Bullish"
            elif bull_points == 2:
                direction = "➡️ Neutral"
            else:
                direction = "📉 Bearish"
            results[name] = {
                "direction": direction,
                "price":     round(current, 2),
                "rsi":       round(rsi, 2),
            }
        except Exception:
            results[name] = "Unknown"
    return results


def scan_sector_direction():
    sector_status = {}
    for sector, ticker in NSE_SECTOR_INDICES.items():
        try:
            df = yf.download(ticker, period="6mo", interval="1d",
                             progress=False, auto_adjust=True)
            if df is None or len(df) < 30:
                sector_status[sector] = {"status": "Data Unavailable", "rsi": 0}
                continue
            df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
            close   = df['Close']
            current = float(close.iloc[-1])
            ema20   = float(compute_ema(close, 20).iloc[-1])
            ema50   = float(compute_ema(close, 50).iloc[-1])
            rsi     = float(compute_rsi(close).iloc[-1])
            macd_l, sig_l, hist = compute_macd(close)
            breakout = current > float(close.iloc[-50:].max() * 0.99)
            bull_points = sum([
                current > ema20,
                current > ema50,
                rsi > 55,
                float(hist.iloc[-1]) > 0,
                breakout,
            ])
            if bull_points >= 3:
                status = "Bullish Breakout"
            elif bull_points == 2:
                status = "Neutral"
            else:
                status = "Bearish"
            sector_status[sector] = {"status": status, "rsi": round(rsi, 2)}
        except Exception:
            sector_status[sector] = {"status": "Error", "rsi": 0}
    return sector_status


# ─────────────────────────────────────────────
# FULL SCAN PIPELINE
# ─────────────────────────────────────────────
def run_full_scan():
    print("\n" + "="*60)
    print("  NSE F&O MARKET SCANNER")
    print(f"  {datetime.now().strftime('%d %b %Y, %I:%M %p')}")
    print("="*60)

    # [0/4] Corporate Actions update
    print("\n[0/4] Updating Corporate Actions...")
    ca_result = update_corporate_actions()
    flagged = get_flagged_symbols()
    flagged_set = {f["symbol"] for f in flagged}
    quarantined = get_quarantined_symbols()
    upcoming_7d = get_upcoming_actions(days_ahead=7)

    print(f"\n[1/4] Scanning NIFTY & SENSEX...")
    market = get_market_direction()

    print("[2/4] Scanning Sectors...")
    sectors = scan_sector_direction()
    bullish_sectors = [s for s, v in sectors.items() if v.get("status") == "Bullish Breakout"]

    print(f"[3/4] Scanning stocks in {len(bullish_sectors)} bullish sectors...")
    if quarantined:
        print(f"   ⚠️  Skipping {len(quarantined)} quarantined symbols: {', '.join(sorted(quarantined))}")

    results = {
        "market":           market,
        "sectors":          sectors,
        "bullish_sectors":  bullish_sectors,
        "breakout_stocks":  {},
        "gfs_stocks":       [],
        "agfs_stocks":      [],
        "must_trade_gfs":   [],
        "must_trade_agfs":  [],
        "corp_actions_new": ca_result["new_risky"],
        "corp_actions_flagged": flagged,
        "corp_actions_upcoming_7d": upcoming_7d,
        "quarantined":      sorted(quarantined),
    }

    all_scanned = {}

    for sector in bullish_sectors:
        stocks = SECTOR_STOCKS.get(sector, [])
        sector_hits = []
        for ticker in stocks:
            if ticker in quarantined:
                continue
            print(f"   Scanning {ticker}...", end="\r")
            if ticker in all_scanned:
                if all_scanned[ticker]:
                    sector_hits.append(all_scanned[ticker])
                continue
            result = scan_stock(ticker, flagged_set)
            if result:
                all_scanned[ticker] = {"sector": sector, **result}
                sector_hits.append(result)
            else:
                all_scanned[ticker] = None
        valid_hits = [h for h in sector_hits if h]
        valid_hits.sort(key=lambda x: int(x["breakout_score"].split("/")[0]), reverse=True)
        if valid_hits:
            results["breakout_stocks"][sector] = valid_hits[:3]

    # All-stocks scan for GFS/AGFS
    print(f"\n[4/4] Scanning remaining stocks for GFS/AGFS...")
    all_sector_stocks = set()
    for stocks in SECTOR_STOCKS.values():
        all_sector_stocks.update(stocks)
    remaining = [s for s in all_sector_stocks if s not in all_scanned and s not in quarantined]

    for ticker in remaining:
        print(f"   Scanning {ticker}...", end="\r")
        result = scan_stock(ticker, flagged_set)
        if result:
            sector = next((s for s, stocks in SECTOR_STOCKS.items() if ticker in stocks), "Other")
            all_scanned[ticker] = {"sector": sector, **result}

    breakout_tickers = set(
        s["ticker"]
        for stocks in results["breakout_stocks"].values()
        for s in stocks
    )

    seen_gfs, seen_agfs = set(), set()
    for ticker, data in all_scanned.items():
        if not data:
            continue
        if data.get("gfs") and ticker not in seen_gfs:
            seen_gfs.add(ticker)
            results["gfs_stocks"].append(data)
            if ticker in breakout_tickers:
                results["must_trade_gfs"].append(data)
        if data.get("agfs") and ticker not in seen_agfs:
            seen_agfs.add(ticker)
            results["agfs_stocks"].append(data)
            if ticker in breakout_tickers:
                results["must_trade_agfs"].append(data)

    return results


# ─────────────────────────────────────────────
# FORMATTER
# ─────────────────────────────────────────────
def format_stock_block(stock):
    gfs_badge  = " | 🎯 GFS"  if stock.get("gfs")  else ""
    agfs_badge = " | ⚡ AGFS" if stock.get("agfs") else ""
    ca_warn    = " | 🚩 CORP ACTION" if stock.get("corp_action_flag") else ""
    badge = gfs_badge + agfs_badge + ca_warn

    lines = [
        f"  📌 {stock['ticker']} @ {stock['current_price']}{badge}",
        f"     Breakout Score : {stock['breakout_score']}",
        f"     RSI (D/W/M)    : {stock['rsi_daily']} / {stock['rsi_weekly']} / {stock['rsi_monthly']}",
        f"     Entry Type     : {stock['entry_type']}",
        f"     Entry Zone     : {stock['entry_zone']}",
        f"     Stop Loss      : {stock['stop_loss']}",
        f"     Target 1       : {stock['target1']}",
        f"     Target 2       : {stock['target2']}",
        f"     Risk:Reward    : 1:{stock['rr_ratio']}",
    ]
    return "\n".join(lines)


def format_results(results):
    lines = []
    lines.append("=" * 60)
    lines.append("📊 NSE F&O MARKET SCANNER REPORT")
    lines.append(datetime.now().strftime("📅 %d %b %Y | ⏰ %I:%M %p"))
    lines.append("=" * 60)

    # Corp action flags FIRST (so user sees them upfront)
    if results.get("corp_actions_flagged"):
        lines.append("\n🚩 CORPORATE ACTION FLAGS (review recommended)")
        lines.append("-" * 60)
        for f in results["corp_actions_flagged"]:
            lines.append(f"  {f['symbol']:<15} | {f['action_type']:<10} | "
                         f"ex-date: {f['ex_date']:<12}")
            lines.append(f"      {f['details'][:80]}")
        lines.append("-" * 60)

    # Upcoming actions in next 7 days (informational, all action types)
    if results.get("corp_actions_upcoming_7d"):
        lines.append("\n📅 UPCOMING CORP ACTIONS (next 7 days)")
        lines.append("-" * 60)
        for u in results["corp_actions_upcoming_7d"]:
            risky_mark = " 🚩" if u.get("is_risky") else ""
            lines.append(f"  {u['ex_date']:<12} | {u['symbol']:<12} | "
                         f"{u['action_type']:<10}{risky_mark}")
            if u.get("details"):
                lines.append(f"      {u['details'][:80]}")
        lines.append("-" * 60)

    if results.get("quarantined"):
        lines.append(f"\n⏸️  QUARANTINED: {', '.join(results['quarantined'])}")

    lines.append("\n🌐 MARKET DIRECTION")
    lines.append("-" * 30)
    for index, data in results["market"].items():
        if isinstance(data, dict):
            lines.append(f"  {index}: {data['direction']}  |  Price: {data['price']}  |  RSI: {data['rsi']}")
        else:
            lines.append(f"  {index}: {data}")

    lines.append("\n📂 SECTOR STATUS")
    lines.append("-" * 30)
    for sector, data in results["sectors"].items():
        if isinstance(data, dict):
            status = data.get("status", "Unknown")
            emoji = {"Bullish Breakout": "🟢", "Neutral": "🟡", "Bearish": "🔴"}.get(status, "⚪")
            lines.append(f"  {emoji} {sector}: {status}  (RSI: {data.get('rsi', 'N/A')})")

    if results["breakout_stocks"]:
        lines.append("\n🚀 BREAKOUT SECTOR STOCKS (Top 3/sector)")
        lines.append("-" * 30)
        for sector, stocks in results["breakout_stocks"].items():
            lines.append(f"\n  [{sector}]")
            for stock in stocks:
                lines.append(format_stock_block(stock))
    else:
        lines.append("\n🚀 BREAKOUT STOCKS: None today")

    if results["gfs_stocks"]:
        lines.append("\n\n🎯 GFS STRATEGY PICKS")
        lines.append("-" * 30)
        for stock in results["gfs_stocks"]:
            lines.append(format_stock_block(stock))
    else:
        lines.append("\n\n🎯 GFS PICKS: None today")

    if results["agfs_stocks"]:
        lines.append("\n\n⚡ AGFS STRATEGY PICKS")
        lines.append("-" * 30)
        for stock in results["agfs_stocks"]:
            lines.append(format_stock_block(stock))
    else:
        lines.append("\n\n⚡ AGFS PICKS: None today")

    if results["must_trade_gfs"] or results["must_trade_agfs"]:
        lines.append("\n\n🔴 MUST TRADE (Breakout + Strategy Overlap)")
        lines.append("-" * 30)
        for stock in results["must_trade_gfs"]:
            lines.append(f"  🔴 [GFS] {stock['ticker']} — {stock['entry_zone']} | SL: {stock['stop_loss']} | T1: {stock['target1']} | RR: 1:{stock['rr_ratio']}")
        for stock in results["must_trade_agfs"]:
            lines.append(f"  🔴 [AGFS] {stock['ticker']} — {stock['entry_zone']} | SL: {stock['stop_loss']} | T1: {stock['target1']} | RR: 1:{stock['rr_ratio']}")
    else:
        lines.append("\n\n🔴 MUST TRADE: None today")

    lines.append("\n" + "=" * 60)
    lines.append("⚠️  Stocks marked 🚩 have recent corp actions; data may be unreliable")
    lines.append("⚠️  For informational purposes only. Trade responsibly.")
    lines.append("=" * 60)

    return "\n".join(lines)


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
if __name__ == "__main__":
    results = run_full_scan()
    report  = format_results(results)
    print("\n\n")
    print(report)

    filename = f"scan_result_{datetime.now().strftime('%d_%b_%Y')}.txt"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(report)
    print(f"\n✅ Report saved to {filename}")



  NSE F&O MARKET SCANNER
  15 May 2026, 07:48 PM

[0/4] Updating Corporate Actions...

📋 Updating Corporate Actions...
   ✅ NSE corp actions fetched: 104 entries
   ✅ 0 new corp actions added
   ⚠️  0 new RISKY events (need review)
   📌 1 total RISKY events pending decision

[1/4] Scanning NIFTY & SENSEX...
[2/4] Scanning Sectors...


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: NIFTY_CHEM.NS"}}}
$NIFTY_CHEM.NS: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['NIFTY_CHEM.NS']: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")


[3/4] Scanning stocks in 4 bullish sectors...
   Scanning ADANIPOWER...
[4/4] Scanning remaining stocks for GFS/AGFS...
   Scanning ACC......R...

$TIPS.NS: possibly delisted; no price data found  (period=2y)

1 Failed download:
['TIPS.NS']: possibly delisted; no price data found  (period=2y)


   Scanning TIPS...

$TIPS.NS: possibly delisted; no price data found  (period=5y)

1 Failed download:
['TIPS.NS']: possibly delisted; no price data found  (period=5y)
$TIPS.NS: possibly delisted; no price data found  (period=10y)

1 Failed download:
['TIPS.NS']: possibly delisted; no price data found  (period=10y)


   Scanning LTIM...D.....

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: LTIM.NS"}}}
$LTIM.NS: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['LTIM.NS']: possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")
$LTIM.NS: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['LTIM.NS']: possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")
$LTIM.NS: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['LTIM.NS']: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")


   Scanning PHOENIXLTD...


📊 NSE F&O MARKET SCANNER REPORT
📅 15 May 2026 | ⏰ 07:50 PM

🚩 CORPORATE ACTION FLAGS (review recommended)
------------------------------------------------------------
  VEDL            | DEMERGER   | ex-date: 30-Apr-2026 
      Demerger
------------------------------------------------------------

📅 UPCOMING CORP ACTIONS (next 7 days)
------------------------------------------------------------
  15-May-2026  | GOPAL        | DIVIDEND  
      Interim Dividend - Re 0.40 Per Share
  15-May-2026  | KRT          | DIVIDEND  
      Distribution - Rs 1.616 Consisting Of Re 0.207 Per Unit As Interest/ Rs 0.981 Pe
  15-May-2026  | ANANDRATHI   | DIVIDEND  
      Dividend - Rs 7 Per Share
  15-May-2026  | SARLAPOLY    | BUYBACK   
      Buy Back
  15-May-2026  | SBIN         | DIVIDEND  
      Dividend - Rs 17.35 Per Share
  15-May-2026  | APTUS        | DIVIDEND  
      Interim Dividend - Rs 2.50 Per Share
  15-May-2026  | IEX          | DIVIDEND  
      Dividend - 